# Lab data analysis (Snowflake, read-only)

One row = one lab test / observation. **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** analysis cells are **SQL cells**, so each result shows Table / Chart / Pivot and the **download** button.

**Grain:** `LabId` should be unique. One patient / encounter can have many lab rows.

**ObservationIdentifier has both views:**
- unique **full names** and how often each occurs
- unique **words** inside those names (`Glucose` counted across `Glucose, Serum`)

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

SQL cells read flat strings (`T`, `C_OBS`, `C_PT`, ...).

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "LAB"   # try LAB, LABS, LAB_RESULT, LABORATORY

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "lab_id": "LabId",
    "encounter_id": "EncounterId/VisitId",
    "patient_id": "Member/PatientId",
    "lab_request_id": "LabRequestId",
    "lab_result_id": "LabResultId",
    "observation_identifier": "ObservationIdentifier",
    "observation_value": "ObservationValue",
    "observation_result_status": "ObservationResultStatus",
    "observation_datetime": "ObservationDateTime",
    "performed_datetime": "PerformedDateTime",
    "analysis_datetime": "AnalysisDateTime",
    "lab_result_note": "LabResultNote",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_LAB = col("lab_id")
C_ENC = col("encounter_id")
C_PT = col("patient_id")
C_REQ = col("lab_request_id")
C_RES = col("lab_result_id")
C_OBS = col("observation_identifier")
C_VAL = col("observation_value")
C_STATUS = col("observation_result_status")
C_DT = col("observation_datetime")
C_PERF = col("performed_datetime")
C_ANL = col("analysis_datetime")
C_NOTE = col("lab_result_note")

print(f"T = {T}")
print(f"C_LAB = {C_LAB}")
print(f"C_ENC = {C_ENC}")
print(f"C_PT = {C_PT}")
print(f"C_REQ = {C_REQ}")
print(f"C_RES = {C_RES}")
print(f"C_OBS = {C_OBS}")
print(f"C_VAL = {C_VAL}")
print(f"C_STATUS = {C_STATUS}")
print(f"C_DT = {C_DT}")
print(f"C_PERF = {C_PERF}")
print(f"C_ANL = {C_ANL}")
print(f"C_NOTE = {C_NOTE}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%LAB%'
     OR UPPER(TABLE_NAME) LIKE '%OBSERVATION%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT *
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_LAB}}) AS UNIQUE_LAB_IDS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_OBS}}) AS UNIQUE_OBSERVATION_IDENTIFIERS,
    COUNT(*) - COUNT(DISTINCT {{C_LAB}}) AS EXTRA_ROWS_VS_UNIQUE_LAB_ID,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_ENC}}), 0), 2) AS AVG_LABS_PER_ENCOUNTER,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_LABS_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_LAB}} IS NULL, 1, 0)) AS NULL_LAB_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_REQ}} IS NULL, 1, 0)) AS NULL_LAB_REQUEST_ID,
    SUM(IFF({{C_RES}} IS NULL, 1, 0)) AS NULL_LAB_RESULT_ID,
    SUM(IFF({{C_OBS}} IS NULL, 1, 0)) AS NULL_OBSERVATION_IDENTIFIER,
    SUM(IFF({{C_VAL}} IS NULL, 1, 0)) AS NULL_OBSERVATION_VALUE,
    SUM(IFF({{C_STATUS}} IS NULL, 1, 0)) AS NULL_RESULT_STATUS,
    SUM(IFF({{C_DT}} IS NULL, 1, 0)) AS NULL_OBSERVATION_DATETIME,
    SUM(IFF({{C_PERF}} IS NULL, 1, 0)) AS NULL_PERFORMED_DATETIME,
    SUM(IFF({{C_ANL}} IS NULL, 1, 0)) AS NULL_ANALYSIS_DATETIME,
    SUM(IFF({{C_NOTE}} IS NULL, 1, 0)) AS NULL_LAB_RESULT_NOTE
FROM {{T}};

## 7. ObservationIdentifier — unique names AND word counts

Two cells:

1. `observation_identifier_counts` — each **full test name** and how often it occurs.
2. `observation_identifier_words` — each **word** inside those names (`Glucose` from `Glucose, Serum`).

In [ ]:
SELECT
    COUNT(DISTINCT {{C_OBS}}) AS UNIQUE_OBSERVATION_IDENTIFIERS,
    SUM(IFF({{C_OBS}} IS NULL, 1, 0)) AS NULL_NAME_ROWS,
    SUM(IFF(TRIM({{C_OBS}}::STRING) = '', 1, 0)) AS BLANK_NAME_ROWS
FROM {{T}};

In [ ]:
SELECT
    {{C_OBS}} AS OBSERVATION_IDENTIFIER,
    COUNT(*) AS LAB_ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LABS
FROM {{T}}
GROUP BY 1
ORDER BY LAB_ROW_COUNT DESC, OBSERVATION_IDENTIFIER;

In [ ]:
WITH tokens AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        TRIM(f.VALUE::STRING) AS WORD
    FROM {{T}},
         LATERAL FLATTEN(
             INPUT => SPLIT(
                 TRIM(REGEXP_REPLACE({{C_OBS}}::STRING, '[^A-Za-z0-9]+', ' ')),
                 ' '
             )
         ) f
    WHERE {{C_OBS}} IS NOT NULL
)
SELECT
    WORD,
    COUNT(*) AS WORD_OCCURRENCES,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_WORD_OCCURRENCES
FROM tokens
WHERE WORD IS NOT NULL
  AND WORD <> ''
GROUP BY 1
ORDER BY WORD_OCCURRENCES DESC, WORD;

## 8. ObservationResultStatus — unique values and counts

In [ ]:
SELECT
    {{C_STATUS}} AS OBSERVATION_RESULT_STATUS,
    COUNT(*) AS LAB_ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LABS
FROM {{T}}
GROUP BY 1
ORDER BY LAB_ROW_COUNT DESC;

## 9. ObservationDateTime distribution vs today

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DT}}) AS MIN_OBSERVATION_DATETIME,
    MAX({{C_DT}}) AS MAX_OBSERVATION_DATETIME,
    SUM(IFF({{C_DT}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_ROWS
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DT}}) AS OBS_YEAR,
    COUNT(*) AS LAB_ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LABS
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DT}} IS NULL THEN 90
            WHEN {{C_DT}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DT}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DT}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DT}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DT}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DT}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DT}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DT}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DT}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 10. Labs per encounter

In [ ]:
WITH per_enc AS (
    SELECT
        {{C_ENC}} AS ENCOUNTER_ID,
        COUNT(*) AS LAB_ROW_COUNT
    FROM {{T}}
    WHERE {{C_ENC}} IS NOT NULL
    GROUP BY 1
)
SELECT
    LAB_ROW_COUNT AS LABS_ON_ENCOUNTER,
    COUNT(*) AS NUMBER_OF_ENCOUNTERS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ENCOUNTERS
FROM per_enc
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_ENC}} AS ENCOUNTER_ID,
    COUNT(*) AS LAB_ROW_COUNT,
    COUNT(DISTINCT {{C_OBS}}) AS UNIQUE_TEST_NAMES,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS
FROM {{T}}
WHERE {{C_ENC}} IS NOT NULL
GROUP BY 1
ORDER BY LAB_ROW_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_PT}} AS PATIENT_ID,
    {{C_LAB}} AS LAB_ID,
    {{C_OBS}} AS OBSERVATION_IDENTIFIER,
    {{C_VAL}} AS OBSERVATION_VALUE,
    {{C_STATUS}} AS OBSERVATION_RESULT_STATUS,
    {{C_DT}} AS OBSERVATION_DATETIME
FROM {{T}}
WHERE {{C_ENC}} = (
        SELECT {{C_ENC}}
        FROM {{T}}
        WHERE {{C_ENC}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_DT}} NULLS LAST, {{C_LAB}};

## Notes

- **Downloading:** run a SQL cell, then use the download arrow on that result grid.
- Full names vs words: `observation_identifier_counts` groups the whole string; `observation_identifier_words` splits it.
- `ObservationValue` is mixed text (`5.2 mmol/L`) and is not grouped here.
- Run `config` before the SQL cells.